# Simple code for Q&A Generation

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
import json

In [11]:
# %pip install ollama

In [4]:
from ollama import chat
from ollama import ChatResponse
import ollama

## For Markdown

In [5]:
corpus_path = "/home/manuel/Job/MELI/Solution/corpus/corpus.jsonl"

# Read file
with open(corpus_path, "r", encoding="utf-8") as file:
    content = file.read()


content_lines = content.splitlines()

In [23]:
def generate_qa(corpus_entry, corp_id):



    # 2. Use a rigid, escaping-safe string for the target schema
    # Small models get confused by template variables inside format strings, so make it a clear, raw JSON example.
    target_format = '{"Question": "extracted question here", "Answer": "extracted answer here"}'

    # 3. Separate your instructions into a System role and a User role
    system_instruction = (
        f"You are a technical Q&A assistant. Analyze the provided JSON metadata and content, "
        f"then generate one relevant question and answer based strictly on the content. "
        f"You MUST respond ONLY with a valid JSON object matching this exact format: {target_format}. "
        # f"Do not include any introductory text, markdown formatting (like ```json), or explanations."
        f"Do not include any relative questions for the versions."
        f"Use all the path, and section information as well as the project name to formulate the question"
    )

    
    entry = json.dumps(corpus_entry)    
    user_message = f"Generate one Q&A pair for this data entry:\n{entry}"

    # 4. Run the chat with format='json' enforced
    resp = ollama.chat(
        model="llama3.1:latest",
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_message},
        ],
        options={"temperature": 0.5},  # Lower temperature makes the model more deterministic
        format="json",  # Force Ollama to output valid JSON
    )

    # 5. Parse and view the output safely
    output_text = resp["message"]["content"]
    try:
        qa_pair = json.loads(output_text)
        qa_pair = {'Id': corp_id, **qa_pair}

        qa_pair = json.dumps(qa_pair, indent=4)

        return qa_pair
    
    except json.JSONDecodeError:
        print("Failed to parse JSON. Raw output:")
        print(output_text)
        return None

In [28]:
n_questions = 2 # Useful for paraphrasing

questions = []
k= len(content_lines)
# k = 3

for i, corp in enumerate(content_lines[0:k]):

    for q in range(n_questions):
       
        id = json.loads(corp)["Id"]
        
        qa_pair = generate_qa(corp, id)

        if qa_pair is not None:

            questions.append(qa_pair)

    if i%20 == 0:
        print(f"Progress: {i*100/k}% ~ {k}")

Progress: 0.0% ~ 2118
Progress: 0.9442870632672332% ~ 2118
Progress: 1.8885741265344664% ~ 2118
Progress: 2.8328611898016995% ~ 2118
Progress: 3.777148253068933% ~ 2118
Progress: 4.721435316336166% ~ 2118
Progress: 5.665722379603399% ~ 2118
Progress: 6.610009442870632% ~ 2118
Progress: 7.554296506137866% ~ 2118
Progress: 8.498583569405099% ~ 2118
Progress: 9.442870632672332% ~ 2118
Progress: 10.387157695939566% ~ 2118
Progress: 11.331444759206798% ~ 2118
Progress: 12.275731822474032% ~ 2118
Progress: 13.220018885741265% ~ 2118
Progress: 14.164305949008499% ~ 2118
Progress: 15.108593012275731% ~ 2118
Progress: 16.052880075542966% ~ 2118
Progress: 16.997167138810198% ~ 2118
Progress: 17.94145420207743% ~ 2118
Progress: 18.885741265344663% ~ 2118
Progress: 19.8300283286119% ~ 2118
Progress: 20.77431539187913% ~ 2118
Progress: 21.718602455146364% ~ 2118
Progress: 22.662889518413596% ~ 2118
Progress: 23.607176581680832% ~ 2118
Progress: 24.551463644948065% ~ 2118
Progress: 25.49575070821529

In [29]:
# Fix
fixed_questions = []

for q in questions:
    fixed_questions.append(json.loads(q.replace("\n    ", "")))

fixed_questions

[{'Id': '2p-revenue-optimizer-api_0',
  'Question': 'What is the purpose of the provided code in the 2p-revenue-optimizer-api project?',
  'Answer': 'The purpose of the provided code is to offer a basic model for JDK 17 / Spring based web applications.'},
 {'Id': '2p-revenue-optimizer-api_0',
  'Question': 'What is the project name of this API?',
  'Answer': '2p-revenue-optimizer-api'},
 {'Id': '2p-revenue-optimizer-api_1',
  'Question': 'What is the path of the 2p-revenue-optimizer-api project?',
  'Answer': '/home/manuel/Job/MELI/docs_raw/2p-revenue-optimizer-api/0.2.2/guide/README.md'},
 {'Id': '2p-revenue-optimizer-api_1',
  'Question': 'What is the contact method for addressing questions and comments regarding the 2p-revenue-optimizer-api project?',
  'Answer': 'Please address any questions and comments to [VendorFlow Issue Tracker](https://github.com/vendorflow/vendorflow/issues).'},
 {'Id': '2p-revenue-optimizer-api_2',
  'Question': 'What does the suffix of each VendorFlow in 2

In [30]:
def save_entries(data, exit_path):

    with open(exit_path, "a", encoding="utf-8") as file:

        for e in data:
            file.write(json.dumps(e) + "\n")

    return

In [31]:
exit_path = "/home/manuel/Job/MELI/Solution/Q&A/qa.jsonl"

save_entries(fixed_questions, exit_path)